# AI Trading Bot — Colab Training (Phase 3)

Run this notebook on Google Colab (T4 or A100 GPU).
**Do NOT run locally** — torch/stable-baselines3 are Colab-only dependencies.

Before running: add `GEMINI_API_KEY` and `NEWS_API_KEY` to Colab Secrets (key icon in left sidebar).

**Recommended execution order:** Cell 1 → 2 → 3 → 4 (Optuna tuning) → 5 (SAC) → 6 (PPO) → 7 (walk-forward) → 8 (compare) → 9 (register)

In [ ]:
# Cell 1 — Setup: clone repo and install training dependencies
!git clone https://github.com/YashKasare21/trading_bot.git
%cd trading_bot
!pip install uv -q
!uv pip install -e '.[training]' -q

In [ ]:
# Cell 2 — Mount Google Drive and load secrets
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['NEWS_API_KEY'] = userdata.get('NEWS_API_KEY')

MODEL_SAVE_DIR = '/content/drive/MyDrive/trading_bot/models/'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f'Model save directory: {MODEL_SAVE_DIR}')

In [ ]:
# Cell 3 — Fetch data and build features
from trading_bot.data.fetcher import MarketDataFetcher
from trading_bot.data.sentiment import SentimentAnalyzer
from trading_bot.features.pipeline import FeaturePipeline
from pathlib import Path
from datetime import date

fetcher = MarketDataFetcher()
df = fetcher.fetch_ohlcv('^NSEI', start=date(2018, 1, 1), end=date(2024, 12, 31))
print(f'Raw OHLCV shape: {df.shape}')

analyzer = SentimentAnalyzer()
sentiment_df = analyzer.fetch_and_score_news('^NSEI', start=date(2018, 1, 1), end=date(2024, 12, 31))
print(f'Sentiment shape: {sentiment_df.shape}')

pipeline = FeaturePipeline(window_size=20, fit_regime=True)
featured_df = pipeline.fit_transform(df, sentiment_df)
pipeline.save(Path(MODEL_SAVE_DIR) / 'pipeline_config.joblib')

print(f'Training data shape: {featured_df.shape}')
print(f'Feature count: {len(pipeline.get_feature_names())}')
print(f'Features: {pipeline.get_feature_names()[:10]} ...')

In [ ]:
# Cell 4 — Run Optuna Tuning (SAC)
# ~45 min on T4 GPU. Skip if you already have best params and set best_sac_config manually.
from trading_bot.models.tune import run_tuning
from pathlib import Path

result = run_tuning(
    featured_df=featured_df,
    algo='SAC',
    n_trials=30,                # 30 trials is enough for a first pass
    n_timesteps_per_trial=75_000,
    storage_path=Path(MODEL_SAVE_DIR) / 'optuna_sac.db',   # resume-able on reconnect
)
print(f"Best SAC Sharpe: {result['best_value']:.3f}")
print(f"Best params: {result['best_params']}")
best_sac_config = result['best_config']

In [ ]:
# Cell 5 — Train SAC with Best Params
from trading_bot.models.train import train_agent
import dataclasses
from pathlib import Path

# Override save_dir to Google Drive
best_sac_config = dataclasses.replace(
    best_sac_config,
    total_timesteps=500_000,
    save_dir=Path(MODEL_SAVE_DIR),
)

sac_result = train_agent(
    config=best_sac_config,
    featured_df=featured_df,
    sentiment_df=sentiment_df,
)
print(f"SAC run: {sac_result['run_name']}")
print(f"Model saved: {sac_result['model_path']}")
print(f"Final eval metrics: {sac_result['final_metrics']}")

In [ ]:
# Cell 6 — Train PPO (for comparison)
from trading_bot.models.train import TrainingConfig, train_agent
from pathlib import Path
from datetime import date

ppo_config = TrainingConfig(
    algo='PPO',
    ticker='^NSEI',
    train_start=date(2018, 1, 1),
    train_end=date(2022, 12, 31),
    total_timesteps=500_000,
    save_dir=Path(MODEL_SAVE_DIR),
)
ppo_result = train_agent(ppo_config, featured_df, sentiment_df)
print(f"PPO run: {ppo_result['run_name']}")
print(f"PPO eval metrics: {ppo_result['final_metrics']}")

In [ ]:
# Cell 7 — Walk-Forward Validation
# Trains one model per window — the only honest way to evaluate a trading model.
# Each window uses a shorter timestep budget to keep total runtime manageable.
from trading_bot.models.walk_forward import run_walk_forward, summarise
import dataclasses
from pathlib import Path

wf_config = dataclasses.replace(
    best_sac_config,
    total_timesteps=150_000,   # shorter per-window budget
    save_dir=Path(MODEL_SAVE_DIR) / 'walk_forward',
)

wf_result = run_walk_forward(
    featured_df=featured_df,
    config=wf_config,
    min_train_years=2,
    test_months=3,
)
print(summarise(wf_result))
print(f'Viable for live trading: {wf_result.is_viable}')

In [ ]:
# Cell 8 — Compare All Strategies
from trading_bot.backtest.engine import BacktestEngine
from pathlib import Path
from datetime import date

engine = BacktestEngine(
    pipeline_path=Path(MODEL_SAVE_DIR) / 'pipeline_config.joblib',
    model_dir=Path(MODEL_SAVE_DIR),
)

test_start = date(2023, 1, 1)
test_end = date(2024, 12, 31)

results = {
    'SAC (Tuned)': engine.run_model(f"{best_sac_config.algo.lower()}_final", 'SAC', '^NSEI', test_start, test_end),
    'PPO': engine.run_model('ppo_final', 'PPO', '^NSEI', test_start, test_end),
    'RSI Rule-Based': engine.run_rule_based('^NSEI', test_start, test_end),
    'Buy & Hold': engine.run_benchmark('^NSEI', test_start, test_end),
}

comparison = engine.compare(results)
print(comparison.to_string())

In [ ]:
# Cell 9 — Register Best Model
from trading_bot.models.registry import ModelRegistry, ModelRecord
from pathlib import Path
from datetime import datetime

registry = ModelRegistry(Path(MODEL_SAVE_DIR) / 'registry.json')

record = ModelRecord(
    run_name=sac_result['run_name'],
    algo='SAC',
    ticker='^NSEI',
    train_start='2018-01-01',
    train_end='2022-12-31',
    total_timesteps=best_sac_config.total_timesteps,
    model_path=sac_result['model_path'],
    vec_norm_path=sac_result['vec_norm_path'] or '',
    pipeline_path=str(Path(MODEL_SAVE_DIR) / 'pipeline_config.joblib'),
    feature_names=pipeline.get_feature_names(),
    hyperparams=result['best_params'],
    train_metrics=sac_result['final_metrics'],
    val_metrics=wf_result.aggregate_metrics,
    created_at=datetime.now().isoformat(),
    notes='Optuna-tuned SAC, walk-forward validated',
    is_production=False,
)
registry.register(record)

# Promote to production only if walk-forward mean Sharpe > 1.0
mean_sharpe = wf_result.aggregate_metrics.get('mean_sharpe', 0)
if mean_sharpe > 1.0:
    registry.promote_to_production(sac_result['run_name'])
    print('Model promoted to PRODUCTION.')
else:
    print(f'Model NOT promoted. Mean Sharpe: {mean_sharpe:.2f} < 1.0 threshold.')

# Confirm registry state
all_models = registry.list_models(ticker='^NSEI')
print(f'Registry now has {len(all_models)} model(s) for ^NSEI')